In [ ]:
import math
from dataclasses import dataclass
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# -----------------------------
# Utilidades de texto
# -----------------------------
def load_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def build_vocab(text: str):
    chars = sorted(list(set(text)))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos

def encode(text: str, stoi: dict) -> torch.Tensor:
    return torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

def decode(ids: List[int], itos: dict) -> str:
    return "".join(itos[i] for i in ids)


# -----------------------------
# Dataset de ventanas
# -----------------------------
class CharWindowDataset(Dataset):
    def __init__(self, data_ids: torch.Tensor, seq_len: int):
        self.data = data_ids
        self.seq_len = seq_len

    def __len__(self):
        return max(0, len(self.data) - self.seq_len - 1)

    def __getitem__(self, idx: int):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y


# -----------------------------
# Modelo baseline: GRU
# -----------------------------
class CharGRULM(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int = 128, hidden_dim: int = 256, num_layers: int = 2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, h0=None):
        emb = self.embed(x)          # (B,T,E)
        out, hN = self.gru(emb, h0)  # (B,T,H)
        logits = self.fc(out)        # (B,T,V)
        return logits, hN


@dataclass
class ExpConfig:
    text_path: str = "texto.txt"
    device: str = "cpu"
    batch_size: int = 64
    emb_dim: int = 128
    hidden_dim: int = 256
    num_layers: int = 2
    lr: float = 3e-4
    epochs: int = 3  # pocas épocas para comparar rápido
    seq_lens: Tuple[int, ...] = (32, 64, 128, 256)
    prompt: str = "La inteligencia artificial "
    temperature: float = 1.0
    gen_tokens: int = 300


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    n_batches = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits, _ = model(x)

        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(1, n_batches)

@torch.no_grad()
def eval_loss(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    n_batches = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits, _ = model(x)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(1, n_batches)

@torch.no_grad()
def generate(model, prompt: str, stoi: dict, itos: dict, max_new_tokens: int, temperature: float, device: str):
    model.eval()
    h = None

    ids = [stoi[ch] for ch in prompt if ch in stoi]
    if not ids:
        ids = [0]

    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    _, h = model(x, h0=h)

    last_id = x[0, -1].item()
    out_ids = ids.copy()

    for _ in range(max_new_tokens):
        inp = torch.tensor([[last_id]], dtype=torch.long, device=device)
        logits, h = model(inp, h0=h)
        logits = logits[0, -1, :] / max(1e-8, temperature)
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()
        out_ids.append(next_id)
        last_id = next_id

    return decode(out_ids, itos)


def run_experiment(cfg: ExpConfig):
    text = load_text(cfg.text_path)
    stoi, itos = build_vocab(text)
    data_ids = encode(text, stoi)

    # split train/val
    n = len(data_ids)
    split = int(0.9 * n)
    train_ids = data_ids[:split]
    val_ids = data_ids[split:]

    device = torch.device(cfg.device)

    results = []  # (L, train_loss, val_loss, ppl, sample)

    print(f"Vocab size: {len(stoi)} | Total chars: {len(data_ids)} | Device: {cfg.device}")

    for L in cfg.seq_lens:
        train_ds = CharWindowDataset(train_ids, seq_len=L)
        val_ds = CharWindowDataset(val_ids, seq_len=L)

        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=True)

        model = CharGRULM(
            vocab_size=len(stoi),
            emb_dim=cfg.emb_dim,
            hidden_dim=cfg.hidden_dim,
            num_layers=cfg.num_layers
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

        print("\n" + "-" * 70)
        print(f"seq_len L = {L} | train_windows={len(train_ds)} | val_windows={len(val_ds)}")

        for epoch in range(1, cfg.epochs + 1):
            tr = train_one_epoch(model, train_loader, optimizer, device)
            vl = eval_loss(model, val_loader, device)
            ppl = math.exp(vl) if vl < 20 else float("inf")
            print(f"  Epoch {epoch:02d} | train_loss={tr:.4f} | val_loss={vl:.4f} | val_ppl={ppl:.2f}")

        sample = generate(
            model, cfg.prompt, stoi, itos,
            max_new_tokens=cfg.gen_tokens,
            temperature=cfg.temperature,
            device=str(device)
        )

        vl = eval_loss(model, val_loader, device)
        ppl = math.exp(vl) if vl < 20 else float("inf")

        results.append((L, tr, vl, ppl, sample))

    return results

In [ ]:
if __name__ == "__main__":
    cfg = ExpConfig(
        text_path="texto.txt",
        device="cpu",           # cambia a "cuda" si tienes GPU
        batch_size=64,
        emb_dim=128,
        hidden_dim=256,
        num_layers=2,
        lr=3e-4,
        epochs=3,               # rápido; sube a 5 si hay tiempo
        seq_lens=(32, 64, 128, 256),
        prompt="La inteligencia artificial ",
        temperature=1.0,
        gen_tokens=300
    )

    results = run_experiment(cfg)

    print("\n" + "=" * 80)
    print("RESUMEN (L vs val_loss / val_ppl)")
    for L, tr, vl, ppl, _ in results:
        print(f"L={L:3d} | val_loss={vl:.4f} | val_ppl={ppl:.2f}")

    print("\n" + "=" * 80)
    print("MUESTRAS (una por L)")
    for L, _, _, _, sample in results:
        print("\n" + "-" * 70)
        print(f"L={L}")
        print(sample[:1200])  # limita impresión